In [1]:
from datetime import timedelta
import ee
import geemap
import os, datetime
import time, math
import geopandas as gpd
from collections import defaultdict
from tqdm import tqdm
import pandas as pd

# Set proxy if needed for Earth Engine / geemap access
# Adjust or remove this line depending on the user's network environment
geemap.set_proxy(port=7890)

# Initialize interactive geemap map
Map = geemap.Map()

# Input shapefile: GRWL river network
# Data source: https://zenodo.org/records/1297434
# Replace with the user-defined path to the downloaded GRWL shapefile
shp_file = "path_to_GRWL_shapefile.shp"

# Read the GRWL shapefile and sort features by OBJECTID
gdf = gpd.read_file(shp_file)
gdf = gdf.sort_values(by='OBJECTID')

In [ ]:
def fmask(image):
    """
    Apply Landsat QA-based masking and scale optical/thermal bands.

    Steps:
    1. Use QA_PIXEL bits to mask clouds, cloud shadows, and snow.
    2. Use QA_RADSAT to mask radiometrically saturated pixels.
    3. Scale optical surface reflectance bands (SR_B*) to reflectance.
    4. Scale thermal bands (ST_B*) to land surface temperature in °C.
    5. Return the masked image with scaled bands replacing the originals.
    """
    cloudsBitMask = (1 << 3)
    cloudshadowBitMask = (1 << 4)
    snowBitMask = (1 << 5)

    qaMask = image.select('QA_PIXEL').bitwiseAnd(cloudsBitMask).eq(0) \
                        .And(image.select('QA_PIXEL').bitwiseAnd(cloudshadowBitMask).eq(0)) \
                        .And(image.select('QA_PIXEL').bitwiseAnd(snowBitMask).eq(0))

    saturationMask = image.select('QA_RADSAT').eq(0)

    # Scale Landsat Collection 2 surface reflectance bands
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)

    # Scale thermal band and convert from Kelvin to Celsius
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0).add(-273.15)

    return image.addBands(opticalBands, None, True) \
                .addBands(thermalBands, None, True) \
                .updateMask(qaMask) \
                .updateMask(saturationMask)


def cal_cloud(image):
    """
    Calculate cloud-related coverage within the target geometry.

    Steps:
    1. Build a QA mask that keeps cloud-free, shadow-free, and snow-free pixels.
    2. Convert it to a cloud mask (1 = masked/cloud-affected pixels).
    3. Compute mean cloud coverage within the study geometry.
    4. Store the result as image property: 'cloud_coverage'.
    """
    cloudsBitMask = (1 << 3)
    cloudshadowBitMask = (1 << 4)
    snowBitMask = (1 << 5)

    qaMask = image.select('QA_PIXEL').bitwiseAnd(cloudsBitMask).eq(0) \
                        .And(image.select('QA_PIXEL').bitwiseAnd(cloudshadowBitMask).eq(0)) \
                        .And(image.select('QA_PIXEL').bitwiseAnd(snowBitMask).eq(0))

    cloudMask = qaMask.lt(1)

    cloudCoverage = cloudMask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e13,
    )

    return image.set('cloud_coverage', cloudCoverage.get('QA_PIXEL'))


def cal_ice_snow(image):
    """
    Estimate ice/snow coverage within the target geometry.

    Steps:
    1. Use blue + green reflectance threshold to identify bright ice/snow pixels.
    2. Combine this spectral mask with the QA-based snow bit.
    3. Compute mean ice/snow coverage within the study geometry.
    4. Store the result as image property: 'ice_snow_coverage'.
    """
    blue = image.select('SR_B1')
    green = image.select('SR_B2')

    # Simple spectral threshold for bright ice/snow surfaces
    ice_snow = blue.add(green).gt(0.45)

    snowBitMask = (1 << 5)
    qaMask = image.select('QA_PIXEL').bitwiseAnd(snowBitMask).eq(0)
    snowMask = qaMask.lt(1)

    # Combine spectral and QA-based snow/ice detection
    ice_snow2 = ice_snow.gt(0).Or(snowMask.gt(0)).rename('j8')

    cal_ice_snow = ice_snow2.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e13,
    )

    return image.set('ice_snow_coverage', cal_ice_snow.get('j8'))


def process_single_mean_image(image):
    """
    Extract median reflectance statistics for one image after water masking.

    Steps:
    1. Apply the AWEIsh-based water mask.
       For simplicity, only the AWEIsh-based water mask is shown in this shared code.
       The full study also considered DSWE-based water extraction; see:
       https://www.mdpi.com/2072-4292/11/4/374
       
    2. Clip the masked image to the target geometry.
    3. Reduce selected bands to median values within the geometry.
    4. Check whether at least one key band has valid output.
    5. If valid, return the original image with median band values stored as properties.
       Otherwise, return None.
    """
    AWEIsh_method = index_l57['AWEIsh'](image).gt(0)

    image_final = image.updateMask(AWEIsh_method).clip(geometry)

    date_ls = ee.Date(image.get('system:time_start'))

    result_median = image_final.select(selectedBands_l57).reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=geometry,
        scale=scale,
        maxPixels=1e13,
    )

    # Use SR_B1 as a simple validity check for the reduced output
    test_value = ee.Number(result_median.get('SR_B1'))

    # Optional combined validity check if another external test value is defined
    both_valid = test_value.neq(None).And(test_value1.neq(None))

    return ee.Algorithms.If(
        test_value,
        ee.Image(image).set(result_median),
        None
    )


def image_to_feature(image):
    """
    Convert an Earth Engine image to a feature using selected properties.

    Output feature properties include:
    - date
    - selected spectral bands in selectedBands_l57
    """
    properties = ee.Dictionary(image.toDictionary()) \
        .select(['date'] + selectedBands_l57)

    return ee.Feature(None, properties)


def process_batch(image_list, start_index, end_index):
    """
    Process a batch of images and return extracted properties as a pandas DataFrame.

    Steps:
    1. Slice the full image list into the requested batch.
    2. Convert the sliced list to an ImageCollection.
    3. Process each image to extract median band statistics.
    4. Convert processed images to features.
    5. Download results to local Python using getInfo().
    6. Convert Earth Engine timestamps to YYYY-MM-DD strings.
    7. Return a pandas DataFrame of image properties.
    """
    current_batch_list = image_list.slice(start_index, end_index)
    current_batch_collection = ee.ImageCollection(current_batch_list)

    final_processed_collection = current_batch_collection.map(process_single_mean_image, True)
    results_feature_collection = final_processed_collection.map(image_to_feature, True)

    current_year_results = results_feature_collection.getInfo()

    properties_list = []

    for feature in current_year_results['features']:
        props = feature['properties']
        date_value = props['date']['value']
        date_str = datetime.datetime.fromtimestamp(date_value / 1000.0).strftime('%Y-%m-%d')
        props['date'] = date_str
        properties_list.append(props)

    return pd.DataFrame(properties_list)


def add_date_property(image):
    """
    Add a formatted date string ('YYYY-MM-dd') as an image property named 'date'.
    """
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
    return image.set('date', date)

In [2]:
# Spatial resolution (m) used for Landsat-based extraction
scale = 30

# Merge Landsat 5 and Landsat 7 Collection 2 Level-2 surface reflectance products
# These two sensors are used here to ensure the best temporal consistency for long-term analysis.
merged = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").merge(
    ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
)

# Water index used in this shared code for river water masking
# For simplicity, only the AWEIsh-based water mask is included here.
# The full study also considered DSWE-based water extraction; see:
# https://www.mdpi.com/2072-4292/11/4/374
index_l57 = {
    'AWEIsh': lambda image: image.expression(
        'BLUE + 2.5 * GREEN - 1.5 * (NIR + SWIR1) - 0.25 * SWIR2', {
            'BLUE': image.select('SR_B1'),
            'GREEN': image.select('SR_B2'),
            'NIR': image.select('SR_B4'),
            'SWIR1': image.select('SR_B5'),
            'SWIR2': image.select('SR_B7')
        }).float().rename('AWEIsh')
}

# Landsat 5/7 bands retained for reflectance extraction
selectedBands_l57 = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']

# Loop through river outlet IDs (example: first 300 features)
for num in tqdm(gdf['OBJECTID'][:300], desc="Processing OBJECTIDs", unit="ID"):
    start_time = time.time()
    results_df = pd.DataFrame()

    # Select the current river geometry from the input GeoDataFrame
    filtered_gdf = gdf[gdf['OBJECTID'] == num]

    # Convert the selected geometry from GeoPandas to Earth Engine format
    try:
        geometry_client = geemap.geopandas_to_ee(filtered_gdf)
    except:
        continue

    # Define extraction buffer width based on river width metadata
    # Minimum width = 100 m; maximum width = 3000 m; fallback = 500 m if missing
    width = min(
        max(
            (500 if pd.isna(filtered_gdf['width_mean'].values[0]) else filtered_gdf['width_mean'].values[0]),
            100
        ),
        3000
    )

    # Buffer the river outlet geometry to define the analysis region
    geometry = geometry_client.geometry().buffer(width)

    # Optional visualization for quality control in geemap
    Map.addLayer(geometry, {'color': 'red'}, 'geometry')
    Map.centerObject(geometry, 13)

    start_time_year = time.time()

    # Build the Landsat image collection for the current river:
    # 1. filter by geometry and date
    # 2. apply QA-based masking and scaling
    # 3. remove scenes with high cloud coverage
    # 4. remove scenes with substantial ice/snow contamination
    # 5. keep selected reflectance bands only
    image_all_year = (
        merged.filterBounds(geometry)
        .filterDate('1984-01-01', '2023-12-31')
        .map(fmask)
        .map(cal_cloud).filter(ee.Filter.lt('cloud_coverage', 0.3))
        .map(cal_ice_snow).filter(ee.Filter.lt('ice_snow_coverage', 0.01))
        .select(selectedBands_l57)
    )

    # Add formatted acquisition date and remove duplicate dates
    image_all_year = image_all_year.map(add_date_property).distinct('date')

    # Count valid images for this river
    image_count = image_all_year.size().getInfo()

    # Skip rivers with no valid Landsat observations
    if image_count == 0:
        continue

    # Convert the image collection to a list for batch processing
    image_list = image_all_year.toList(image_count)

    all_properties_list = []

    # Process images in batches to avoid memory or server-side limits
    batch_size = 100
    num_batches = (image_count + batch_size - 1) // batch_size

    for i in range(num_batches):
        start_index = i * batch_size
        end_index = min((i + 1) * batch_size, image_count)

        try:
            df_batch = process_batch(image_list, start_index, end_index)
            all_properties_list.append(df_batch)
        except:
            continue

    try:
        # Merge all batch outputs into one table
        final_df = pd.concat(all_properties_list, ignore_index=True)

        # Add the river ID and sort records by acquisition date
        final_df['ID'] = num
        final_df = final_df.sort_values(by='date')

        # Print simple processing summary:
        # river ID, total valid images, extracted records, elapsed time
        iteration_time = time.time() - start_time
        print(num, '\t', image_count, '\t', len(final_df), '\t', round(iteration_time, 2), end='\n')

        # User-defined output directory for saving per-river extraction results
        output_dir = "path_to_output_directory"

        # Save extracted reflectance statistics to CSV
        final_df.to_csv(
            rf"{output_dir}\{num}_{len(final_df)/image_count*100:.0f}%_{len(final_df)}_{image_count}.csv",
            index=False
        )

    except:
        continue

Processing OBJECTIDs:   1%|▌                                                        | 1/100 [01:27<2:24:38, 87.66s/ID]

206965 	 261 	 234 	 87.65


Processing OBJECTIDs:   2%|█                                                       | 2/100 [03:45<3:11:01, 116.96s/ID]

206966 	 284 	 252 	 137.46


Processing OBJECTIDs:   3%|█▋                                                      | 3/100 [06:50<3:59:33, 148.18s/ID]

...

207062 	 493 	 411 	 252.47


Processing OBJECTIDs:  99%|██████████████████████████████████████████████████████▍| 99/100 [3:55:28<05:03, 303.57s/ID]

207063 	 597 	 419 	 357.41


Processing OBJECTIDs: 100%|██████████████████████████████████████████████████████| 100/100 [4:02:05<00:00, 145.26s/ID]

207064 	 611 	 506 	 396.69
